In [1]:
# heart_disease_DecisionTree_pipeline_repeat1000.py

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

DATA_FILE = "heart_disease.csv"
TARGET_COLUMN = "target"

# Choose one criterion:
# "gini", "entropy", "log_loss", "entropy_alpha", "gini_beta", "proposed"
CRITERION = "proposed"

ALPHA = 0.6   # 0 <= alpha <= 1
BETA = 0.5    # 0 <= beta <= 0.5
GAMMA = 0.8   # 0 <= gamma <= 1

REPEATS = 1  # Change to 1000 for the full experiment
OUTER_FOLDS = 10
INNER_FOLDS = 10
TEST_SIZE = 0.20
RANDOM_STATE = 42

VALID_CRITERIA = {
    "gini", "entropy", "log_loss",
    "entropy_alpha", "gini_beta", "proposed",
}

if CRITERION not in VALID_CRITERIA:
    raise ValueError(f"Unknown criterion: {CRITERION}")

# Load data
data = pd.read_csv(DATA_FILE)
X_raw = data.drop(columns=[TARGET_COLUMN])
y_raw = data[TARGET_COLUMN]

if not pd.api.types.is_numeric_dtype(y_raw):
    encoder = LabelEncoder()
    y_raw = pd.Series(
        encoder.fit_transform(y_raw.astype(str)),
        index=y_raw.index,
        name=TARGET_COLUMN,
    )

if y_raw.nunique() != 2:
    raise ValueError("This script expects a binary target.")


def fit_preprocessor(X):
    X = X.copy()
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    numeric_imputer = None
    categorical_imputer = None
    category_maps = {}

    if numeric_cols:
        numeric_imputer = SimpleImputer(strategy="mean")
        X[numeric_cols] = numeric_imputer.fit_transform(X[numeric_cols])

    if categorical_cols:
        categorical_imputer = SimpleImputer(strategy="most_frequent")
        X[categorical_cols] = categorical_imputer.fit_transform(X[categorical_cols])

    if "age" in X.columns:
        X["age_binned"] = pd.cut(
            X["age"],
            bins=[0, 40, 50, 60, 70, 100],
            labels=False,
            include_lowest=True,
        )
        X = X.drop(columns=["age"])

    columns_to_encode = [
        col for col in X.columns
        if X[col].dtype == object
        or str(X[col].dtype).startswith("category")
        or X[col].nunique(dropna=False) < 10
    ]

    for col in columns_to_encode:
        values = X[col].astype(str)
        classes = sorted(values.unique())
        mapping = {value: index for index, value in enumerate(classes)}
        category_maps[col] = mapping
        X[col] = values.map(mapping).astype(float)

    state = {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "numeric_imputer": numeric_imputer,
        "categorical_imputer": categorical_imputer,
        "category_maps": category_maps,
        "output_columns": X.columns.tolist(),
    }
    return X.astype(float), state


def transform_preprocessor(X, state):
    X = X.copy()

    if state["numeric_cols"]:
        X[state["numeric_cols"]] = state["numeric_imputer"].transform(
            X[state["numeric_cols"]]
        )

    if state["categorical_cols"]:
        X[state["categorical_cols"]] = state["categorical_imputer"].transform(
            X[state["categorical_cols"]]
        )

    if "age" in X.columns:
        X["age_binned"] = pd.cut(
            X["age"],
            bins=[0, 40, 50, 60, 70, 100],
            labels=False,
            include_lowest=True,
        )
        X = X.drop(columns=["age"])

    for col, mapping in state["category_maps"].items():
        X[col] = X[col].astype(str).map(mapping).fillna(-1).astype(float)

    X = X.reindex(columns=state["output_columns"], fill_value=0.0)
    return X.astype(float)


def build_model(random_state):
    params = {
        "criterion": CRITERION,
        "random_state": random_state,
    }

    if CRITERION == "entropy_alpha":
        params["alpha"] = ALPHA
    elif CRITERION == "gini_beta":
        params["beta"] = BETA
    elif CRITERION == "proposed":
        params.update(alpha=ALPHA, beta=BETA, gamma=GAMMA)

    return DecisionTreeClassifier(**params)


def compute_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0


def positive_class_probability(model, X):
    class_index = np.where(model.classes_ == 1)[0]
    if class_index.size != 1:
        raise ValueError("Class label 1 was not found in the fitted model.")
    return model.predict_proba(X)[:, class_index[0]]


X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_raw,
    y_raw,
    test_size=TEST_SIZE,
    stratify=y_raw,
    random_state=RANDOM_STATE,
)

results = []

for r in range(REPEATS):
    outer_cv = StratifiedKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=r,
    )

    outer_acc, outer_sens, outer_spec, outer_f1, outer_auc = [], [], [], [], []
    outer_inner_acc = []

    for outer_fold, (train_idx, val_idx) in enumerate(
        outer_cv.split(X_train_full, y_train_full),
        start=1,
    ):
        X_outer_train = X_train_full.iloc[train_idx].copy()
        y_outer_train = y_train_full.iloc[train_idx].copy()
        X_outer_val = X_train_full.iloc[val_idx].copy()
        y_outer_val = y_train_full.iloc[val_idx].copy()

        inner_cv = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=r,
        )
        inner_acc = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            X_outer_train,
            y_outer_train,
        ):
            X_inner_train_raw = X_outer_train.iloc[inner_train_idx].copy()
            y_inner_train = y_outer_train.iloc[inner_train_idx].copy()
            X_inner_val_raw = X_outer_train.iloc[inner_val_idx].copy()
            y_inner_val = y_outer_train.iloc[inner_val_idx].copy()

            X_inner_train, prep_state = fit_preprocessor(X_inner_train_raw)
            X_inner_val = transform_preprocessor(X_inner_val_raw, prep_state)

            model = build_model(r * OUTER_FOLDS + outer_fold)
            model.fit(X_inner_train, y_inner_train)
            y_inner_pred = model.predict(X_inner_val)
            inner_acc.append(accuracy_score(y_inner_val, y_inner_pred))

        outer_inner_acc.append(np.mean(inner_acc))

        X_outer_train_prep, prep_state = fit_preprocessor(X_outer_train)
        X_outer_val_prep = transform_preprocessor(X_outer_val, prep_state)

        model = build_model(r * OUTER_FOLDS + outer_fold)
        model.fit(X_outer_train_prep, y_outer_train)

        y_outer_pred = model.predict(X_outer_val_prep)
        y_outer_prob = positive_class_probability(model, X_outer_val_prep)

        outer_acc.append(accuracy_score(y_outer_val, y_outer_pred))
        outer_sens.append(recall_score(y_outer_val, y_outer_pred, zero_division=0))
        outer_spec.append(compute_specificity(y_outer_val, y_outer_pred))
        outer_f1.append(f1_score(y_outer_val, y_outer_pred, zero_division=0))
        outer_auc.append(roc_auc_score(y_outer_val, y_outer_prob))

    X_train_final, final_prep_state = fit_preprocessor(X_train_full)
    X_test_final = transform_preprocessor(X_test, final_prep_state)

    final_model = build_model(r)
    final_model.fit(X_train_final, y_train_full)

    y_test_pred = final_model.predict(X_test_final)
    y_test_prob = positive_class_probability(final_model, X_test_final)

    results.append({
        "Repeat": r,
        "Criterion": CRITERION,
        "Alpha": ALPHA if CRITERION in {"entropy_alpha", "proposed"} else np.nan,
        "Beta": BETA if CRITERION in {"gini_beta", "proposed"} else np.nan,
        "Gamma": GAMMA if CRITERION == "proposed" else np.nan,
        "Mean Inner Accuracy": np.mean(outer_inner_acc),
        "Outer CV Accuracy": np.mean(outer_acc),
        "Outer CV Sensitivity": np.mean(outer_sens),
        "Outer CV Specificity": np.mean(outer_spec),
        "Outer CV F1": np.mean(outer_f1),
        "Outer CV AUC": np.mean(outer_auc),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Test Sensitivity": recall_score(y_test, y_test_pred, zero_division=0),
        "Test Specificity": compute_specificity(y_test, y_test_pred),
        "Test F1": f1_score(y_test, y_test_pred, zero_division=0),
        "Test AUC": roc_auc_score(y_test, y_test_prob),
    })

    print(f"Completed repeat {r + 1}/{REPEATS} using criterion={CRITERION!r}")

results_df = pd.DataFrame(results)

metrics = [
    "Mean Inner Accuracy",
    "Outer CV Accuracy",
    "Outer CV Sensitivity",
    "Outer CV Specificity",
    "Outer CV F1",
    "Outer CV AUC",
    "Test Accuracy",
    "Test Sensitivity",
    "Test Specificity",
    "Test F1",
    "Test AUC",
]

excel_file = f"heart_disease_DecisionTree_{CRITERION}_repeat{REPEATS}.xlsx"

with pd.ExcelWriter(excel_file, engine="xlsxwriter") as writer:
    results_df.to_excel(writer, sheet_name="Metrics", index=False)
    workbook = writer.book

    for index, metric in enumerate(metrics, start=1):
        fig = plt.figure(figsize=(6, 4))
        plt.hist(results_df[metric].dropna(), bins=30, orientation="horizontal")
        plt.xlabel("Frequency")
        plt.ylabel(metric)
        plt.title(f"Histogram of {metric}")
        plt.tight_layout()

        img_path = f"{CRITERION}_{index:02d}_{metric.replace(' ', '_')}.png"
        plt.savefig(img_path, dpi=150)
        plt.close(fig)

        worksheet = workbook.add_worksheet(f"{index:02d}_{metric[:25]}")
        worksheet.insert_image("B2", img_path)

for index, metric in enumerate(metrics, start=1):
    img_path = f"{CRITERION}_{index:02d}_{metric.replace(' ', '_')}.png"
    if os.path.exists(img_path):
        os.remove(img_path)

print(f"All done. Results and histograms saved to {excel_file}")

ImportError: No module named 'sklearn.__check_build._check_build'
___________________________________________________________________________
Contents of C:\Users\al_ab\Downloads\scikit-learn\scikit-learn\sklearn\__check_build:
meson.build               _check_build.pyx          __init__.py
__pycache__
___________________________________________________________________________
It seems that scikit-learn has not been built correctly.

If you have installed scikit-learn from source, please do not forget
to build the package before using it. For detailed instructions, see:
https://scikit-learn.org/dev/developers/development_setup.html#install-editable-version-of-scikit-learn

If you have used an installer, please check that it is suited for your
Python version, your operating system and your platform.